# Build a validated NVIDIA reranking stage around a local retrieval experiment

Separate retrieval quality from the API contract of a hosted reranker. Construct synthetic support passages, fuse candidates across collections, validate index-based ranking responses, and measure ordering on authored relevance labels. The default uses an explicitly synthetic response for contract testing; an optional NVIDIA HTTP branch performs real reranking only when you enable it.

**Prerequisites:** Python 3.12 or later and the tested environment in the [collection README](../../../README.md). Install the [root requirements](../../../requirements.txt) into an isolated environment before opening this notebook; select that environment's kernel. Run all cells from a fresh kernel in order. This notebook makes no network calls, reads no external files, and writes no artifacts by default.

**Reproducibility:** all examples and data here are synthetic. Simulator and sampling seeds are explicit. Assertions distinguish mathematical identities, finite-shot checks, and integration contracts. Approximate comparisons state their tolerance.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Preserve identity across the pipeline

A reranking response often references positions in a request. Treat those positions as request-local identifiers; never attach a score to a passage by guessing which list ordering the server used. Keep a stable document ID in metadata, deduplicate before calling the model, and validate outputs before displaying retrieved evidence.


In [ ]:
import math
import os
import re
from collections import defaultdict

import pandas as pd

query = "How should I replace an air filter safely?"
passages = [
    {
        "id": "guide-a",
        "collection": "manuals",
        "text": "Before replacing an air filter, switch off the ventilation unit and disconnect its power supply.",
    },
    {
        "id": "guide-b",
        "collection": "manuals",
        "text": "Match the replacement air filter size and airflow arrow to the equipment specification.",
    },
    {
        "id": "note-c",
        "collection": "operations",
        "text": "The air filter replacement training starts on Thursday. Bring a notebook.",
    },
    {
        "id": "note-d",
        "collection": "operations",
        "text": "After isolating power, remove the old air filter, fit the new filter, and secure the cover before restarting.",
    },
    {
        "id": "paper-e",
        "collection": "research",
        "text": "Filter pressure drop rises as particles accumulate. Replace a loaded filter according to the service interval.",
    },
    {
        "id": "paper-f",
        "collection": "research",
        "text": "A color filter in photography changes the light spectrum entering a camera.",
    },
]
judgments = {"guide-a": 3, "note-d": 3, "guide-b": 2, "paper-e": 1}


def tokenize(text):
    return set(re.findall(r"[a-z]+", text.lower()))


def retrieve(collection, k=2):
    scored = [
        (d, len(tokenize(query) & tokenize(d["text"])))
        for d in passages
        if d["collection"] == collection
    ]
    return sorted(scored, key=lambda pair: (-pair[1], pair[0]["id"]))[:k]


rrf = defaultdict(float)
for collection in sorted({d["collection"] for d in passages}):
    for rank, (doc, _) in enumerate(retrieve(collection), 1):
        rrf[doc["id"]] += 1 / (60 + rank)
lookup = {d["id"]: d for d in passages}
candidates = [lookup[doc_id] for doc_id in sorted(rrf, key=lambda doc_id: (-rrf[doc_id], doc_id))]
assert len(candidates) == len({doc["id"] for doc in candidates})
print(pd.DataFrame(candidates))

## 2. Exercise the response boundary with synthetic scores

The fixture below is authored test data, **not a captured NVIDIA response and not a model-quality result**. It intentionally arrives out of order. A robust adapter rejects duplicate or out-of-range indices, booleans masquerading as integer indices, and nonfinite scores. Scores are ranking values, not calibrated probabilities. Partial responses are allowed because a service may return only top-N candidates.


In [ ]:
def validate_rankings(rankings, documents, top_n=3):
    if type(top_n) is not int or not 1 <= top_n <= len(documents):
        raise ValueError("Invalid top_n")
    seen = set()
    validated = []
    for item in rankings:
        index = item.get("index")
        score = item.get("logit")
        if type(index) is not int or not 0 <= index < len(documents) or index in seen:
            raise ValueError("Invalid or duplicate response index")
        if type(score) not in (int, float) or not math.isfinite(score):
            raise ValueError("Score must be finite and numeric")
        seen.add(index)
        validated.append({**documents[index], "score": float(score)})
    return sorted(validated, key=lambda doc: (-doc["score"], doc["id"]))[:top_n]


fixture_scores = {
    "guide-a": 4.4,
    "note-d": 4.2,
    "guide-b": 2.1,
    "paper-e": 0.8,
    "note-c": -0.5,
    "paper-f": -1.3,
}
fixture = [
    {"index": i, "logit": fixture_scores[d["id"]]} for i, d in reversed(list(enumerate(candidates)))
]
ranked = validate_rankings(fixture, candidates)
assert [d["id"] for d in ranked] == ["guide-a", "note-d", "guide-b"]
for bad in [
    [{"index": -1, "logit": 1}],
    [{"index": 0, "logit": float("nan")}],
    [{"index": True, "logit": 1}],
    [{"index": 0, "logit": 1}, {"index": 0, "logit": 2}],
]:
    try:
        validate_rankings(bad, candidates)
    except ValueError:
        pass
    else:
        raise AssertionError("Malformed response was accepted")
print(pd.DataFrame(ranked)[["id", "score", "text"]])

## 3. Distinguish fixture tests from relevance evaluation

For the authored labels, compute candidate recall and NDCG@3 before and after the fixture order. The fixture deliberately puts the highest grades first, so its NDCG is one by construction. That is a test of metric and mapping code, not evidence that NVIDIA or any other model improves search. A real study must freeze judgments first, run an unseen query set through the provider, and compare latency, cost, recall, and ranking quality.


In [ ]:
def ndcg(ids, k=3):
    def dcg(grades):
        return sum((2**grade - 1) / math.log2(i + 2) for i, grade in enumerate(grades))

    denominator = dcg(sorted(judgments.values(), reverse=True)[:k])
    return dcg([judgments.get(doc_id, 0) for doc_id in ids[:k]]) / denominator


candidate_ids = [d["id"] for d in candidates]
recall = len(set(candidate_ids) & judgments.keys()) / len(judgments)
assert recall == 1 and abs(ndcg([d["id"] for d in ranked]) - 1) < 1e-12
print(
    {
        "candidate_recall": recall,
        "lexical_RRF_NDCG@3": ndcg(candidate_ids),
        "authored_fixture_NDCG@3": ndcg([d["id"] for d in ranked]),
    }
)

## 4. An explicit NVIDIA HTTP adapter

The [official hosted reranking contract](https://docs.api.nvidia.com/nim/reference/nvidia-nv-rerankqa-mistral-4b-v3-infer) accepts a model ID, `query.text`, `passages[].text`, and a truncation policy. Its response contains `rankings[]` with zero-based `index` and numeric `logit` ([response semantics](https://docs.nvidia.com/nim/nemo-retriever/text-reranking/latest/using-reranking.html)). Keep `RUN_NVIDIA=False` for offline execution. The audited core environment already includes HTTPX.

This small adapter fixes the official HTTPS endpoint, enables certificate checks, disables redirects and environment proxy configuration, and bounds connection/read waits. Model IDs are body data, never URL paths. The returned indices must cover every submitted passage exactly once; we then select the top three. Provider status failures are reported without echoing response bodies or credentials. There are no automatic retries. The context manager closes the transport after every call.

The request boundary also suppresses raw transport/protocol exceptions: these can echo a malformed credential header even before a valid HTTP response arrives. Local argument-validation errors remain visible, and HTTP-status failures retain only the numeric status.


In [ ]:
import httpx

NVIDIA_RERANK_URL = "https://ai.api.nvidia.com/v1/retrieval/nvidia/reranking"


def nvidia_rank(query, documents, model, api_key, *, transport=None):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query is required")
    if not isinstance(model, str) or not 1 <= len(model) <= 128:
        raise ValueError("A model ID is required")
    if not isinstance(api_key, str) or not api_key.strip():
        raise ValueError("An API key is required")
    if not 1 <= len(documents) <= 512:
        raise ValueError("Expected 1 to 512 passages")
    texts = [doc["text"] for doc in documents]
    if any(not isinstance(text, str) or not text.strip() for text in texts):
        raise ValueError("Empty passage")
    payload = {
        "model": model,
        "query": {"text": query},
        "passages": [{"text": text} for text in texts],
        "truncate": "NONE",
    }
    # transport is solely a test seam; the live caller leaves it unset.
    try:
        with httpx.Client(
            timeout=httpx.Timeout(30.0, connect=10.0),
            verify=True,
            follow_redirects=False,
            trust_env=False,
            transport=transport,
        ) as client:
            response = client.post(
                NVIDIA_RERANK_URL, headers={"Authorization": "Bearer " + api_key}, json=payload
            )
    except Exception:
        # A malformed credential can be echoed by header/protocol exceptions.
        raise RuntimeError("NVIDIA transport failed; no response was verified") from None
    if response.status_code != 200:
        raise RuntimeError(f"NVIDIA reranking failed: HTTP {response.status_code}")
    try:
        body = response.json()
    except ValueError:
        raise ValueError("Provider returned invalid JSON") from None
    rows = body.get("rankings") if isinstance(body, dict) else None
    if (
        not isinstance(rows, list)
        or len(rows) != len(documents)
        or any(not isinstance(row, dict) for row in rows)
    ):
        raise ValueError("Expected one ranking object per passage")
    return validate_rankings(rows, documents, top_n=min(3, len(documents)))

## 5. Verify the request/response contract without a network

The mock returns the same explicitly fabricated rankings used earlier. These assertions test serialization, global ID mapping, and rejection of HTTP errors, redirects, malformed JSON, missing rankings, and duplicate indices. They do not test NVIDIA model availability, credentials, service compatibility, or retrieval quality. In particular, a successful mock is not evidence of a successful hosted call.


In [ ]:
import json
import traceback

import h11

seen_requests = []


def contract_response(request):
    assert request.method == "POST" and str(request.url) == NVIDIA_RERANK_URL
    assert request.headers["Authorization"] == "Bearer synthetic-test-key"
    body = json.loads(request.content)
    assert body == {
        "model": "synthetic-model-contract",
        "query": {"text": query},
        "passages": [{"text": doc["text"]} for doc in candidates],
        "truncate": "NONE",
    }
    seen_requests.append(request.url.host)
    return httpx.Response(200, json={"rankings": fixture})


checked = nvidia_rank(
    query,
    candidates,
    "synthetic-model-contract",
    "synthetic-test-key",
    transport=httpx.MockTransport(contract_response),
)
assert seen_requests == ["ai.api.nvidia.com"]
assert [doc["id"] for doc in checked] == ["guide-a", "note-d", "guide-b"]
for status, body in [
    (401, {}),
    (302, {}),
    (200, {}),
    (200, {"rankings": [{"index": 0, "logit": 1}] * len(candidates)}),
]:
    requests_seen = []

    def reject_response(request):
        requests_seen.append(request.url.host)
        return httpx.Response(
            status, json=body, headers={"Location": "https://invalid.example/redirect"}
        )

    try:
        nvidia_rank(
            query,
            candidates,
            "synthetic-model-contract",
            "synthetic-test-key",
            transport=httpx.MockTransport(reject_response),
        )
    except (ValueError, RuntimeError):
        pass
    else:
        raise AssertionError("Invalid provider response accepted")
    assert requests_seen == ["ai.api.nvidia.com"], "A redirect was followed"
try:
    nvidia_rank(
        query,
        candidates,
        "synthetic-model-contract",
        "synthetic-test-key",
        transport=httpx.MockTransport(lambda request: httpx.Response(200, text="invalid-json")),
    )
except ValueError:
    pass
else:
    raise AssertionError("Malformed JSON accepted")
print("Offline HTTP contract and response-rejection checks passed; no hosted requests sent.")
# Exercise HTTPX's audited HTTP/1.1 header validator without network access.


def reject_malformed_headers(request):
    try:
        h11.Request(method=request.method, target=request.url.raw_path, headers=request.headers.raw)
    except h11.LocalProtocolError as error:
        raise httpx.LocalProtocolError(str(error), request=request) from error
    raise AssertionError("Malformed credential unexpectedly passed wire validation")


bad_key = "synthetic-private-token-sentinel\n"
try:
    nvidia_rank(
        query,
        candidates,
        "synthetic-model-contract",
        bad_key,
        transport=httpx.MockTransport(reject_malformed_headers),
    )
except RuntimeError as error:
    assert str(error) == "NVIDIA transport failed; no response was verified"
    assert bad_key.strip() not in "".join(traceback.format_exception(error))
    assert error.__suppress_context__
else:
    raise AssertionError("Malformed credential was accepted")
print("Malformed-header errors are sanitized before rendering notebook output.")

## 6. Optional: perform a hosted ranking

Before opting in, set `NVIDIA_API_KEY` and `NVIDIA_RERANK_MODEL` to an account-authorized model supported by the fixed endpoint above. Check current model availability in the linked API reference; this notebook intentionally does not guess an available model. Review the provider's data terms and charges. Only the synthetic query and passages are submitted. `truncate='NONE'` requests rejection rather than silent truncation for oversized inputs; account/model limits still require a live check. Ambient `NVIDIA_BASE_URL` is ignored.


In [ ]:
RUN_NVIDIA = False
if RUN_NVIDIA:
    if not os.environ.get("NVIDIA_API_KEY") or not os.environ.get("NVIDIA_RERANK_MODEL"):
        raise ValueError("Set NVIDIA_API_KEY and NVIDIA_RERANK_MODEL explicitly")
    output = nvidia_rank(
        query, candidates, os.environ["NVIDIA_RERANK_MODEL"], os.environ["NVIDIA_API_KEY"]
    )
    output_ids = [doc["id"] for doc in output]
    print(pd.DataFrame(output)[["id", "score"]])
    print("NVIDIA NDCG@3 on this single synthetic query:", ndcg(output_ids))
else:
    print("NVIDIA integration was not invoked. Completed response-contract and metric tests only.")

## Limits, cleanup, and next steps

A reranker cannot retrieve excluded evidence, enforce document permissions, or guarantee factual answers. Apply authorization before candidate selection and again before displaying results. Scores depend on the model and query; avoid transferring a numerical threshold across model versions. Truncation and maximum request sizes require separate tests with your selected model.

The default creates no remote resources. The optional branch performs inference only and prints document IDs/scores, never credentials. Clear credentials from the process environment after use. A vector database or NVIDIA embedding service can replace first-stage lexical retrieval, but neither is exercised here.

Primary references: [NVIDIA Retrieval APIs](https://docs.api.nvidia.com/nim/reference/retrieval-apis), [NVIDIA reranking request/response contract](https://docs.api.nvidia.com/nim/reference/nvidia-nv-rerankqa-mistral-4b-v3-infer).
